### new special tokens


- DeepSeek V3.1 add four new special tokens (compared to V3-base)
    - `<｜search▁begin｜>` (id: 128796)
    - `<｜search▁end｜>` (id: 128797)
        - 网页端默认开启搜索，既使为选中搜索按钮
        - 目前官方的 huggingface 版本，在 chat template 中没有关于这对 special token 的处理
    - `<think>` (id: 128798)
    - `</think>` (id: 128799)
        - 之前只有 R1 模型才有这个 special token

```
{% if not add_generation_prompt is defined %}
    {% set add_generation_prompt = false %}
{% endif %}

{% if not thinking is defined %}
    {% set thinking = false %}
{% endif %}

{% set ns = namespace(is_first=false, is_tool=false, system_prompt='', is_first_sp=true, is_last_user=false) %}

{# Part 1: Construct the system prompt by concatenating all system messages #}
{%- for message in messages %}
    {%- if message['role'] == 'system' %}
        {%- if ns.is_first_sp %}
            {% set ns.system_prompt = ns.system_prompt + message['content'] %}
            {% set ns.is_first_sp = false %}
        {%- else %}
            {% set ns.system_prompt = ns.system_prompt + '\n\n' + message['content'] %}
        {%- endif %}
    {%- endif %}
{%- endfor %}

{{ bos_token }}
{{ ns.system_prompt }}

{# Part 2: Process all messages (user, assistant, tool) in order #}
{%- for message in messages %}
    
    {# Handle 'user' role #}
    {%- if message['role'] == 'user' %}
        {%- set ns.is_tool = false -%}
        {%- set ns.is_first = false -%}
        {%- set ns.is_last_user = true -%}
        {{ '<｜User｜>' + message['content'] }}
    {%- endif %}

    {# Handle 'assistant' role with tool calls #}
    {%- if message['role'] == 'assistant' and message['tool_calls'] is defined and message['tool_calls'] is not none %}
        {%- if ns.is_last_user %}
            {{ '<｜Assistant｜></think>' }}
        {%- endif %}
        {%- set ns.is_last_user = false -%}
        {%- set ns.is_first = false %}
        {%- set ns.is_tool = false -%}
        {%- for tool in message['tool_calls'] %}
            {%- if not ns.is_first %}
                {%- if message['content'] is none %}
                    {{ '<｜tool calls begin｜><｜tool call begin｜>'+ tool['function']['name'] + '<｜tool sep｜>' + tool['function']['arguments'] + '<｜tool call end｜>' }}
                {%- else %}
                    {{ message['content'] + '<｜tool calls begin｜><｜tool call begin｜>' + tool['function']['name'] + '<｜tool sep｜>' + tool['function']['arguments'] + '<｜tool call end｜>' }}
                {%- endif %}
                {%- set ns.is_first = true -%}
            {%- else %}
                {{ '<｜tool call begin｜>'+ tool['function']['name'] + '<｜tool sep｜>' + tool['function']['arguments'] + '<｜tool call end｜>' }}
            {%- endif %}
        {%- endfor %}
        {{ '<｜tool calls end｜><｜end of sentence｜>' }}
    {%- endif %}

    {# Handle 'assistant' role without tool calls (regular message) #}
    {%- if message['role'] == 'assistant' and (message['tool_calls'] is not defined or message['tool_calls'] is none) %}
        {%- if ns.is_last_user %}
            {{ '<｜Assistant｜>' }}
            {%- if message['prefix'] is defined and message['prefix'] and thinking %}
                {{ '<think>' }}  
            {%- else %}
                {{ '</think>' }}
            {%- endif %}
        {%- endif %}
        {%- set ns.is_last_user = false -%}
        {%- if ns.is_tool %}
            {{ message['content'] + '<｜end of sentence｜>' }}
            {%- set ns.is_tool = false -%}
        {%- else %}
            {%- set content = message['content'] -%}
            {%- if '</think>' in content %}
                {%- set content = content.split('</think>', 1)[1] -%}
            {%- endif %}
            {{ content + '<｜end of sentence｜>' }}
        {%- endif %}
    {%- endif %}

    {# Handle 'tool' role #}
    {%- if message['role'] == 'tool' %}
        {%- set ns.is_last_user = false -%}
        {%- set ns.is_tool = true -%}
        {{ '<｜tool output begin｜>' + message['content'] + '<｜tool output end｜>' }}
    {%- endif %}

{%- endfor -%}

{# Part 3: Add the generation prompt at the end if needed #}
{%- if add_generation_prompt and ns.is_last_user and not ns.is_tool %}
    {{ '<｜Assistant｜>' }}
    {%- if not thinking %}
        {{ '</think>' }}
    {%- else %}
        {{ '<think>' }}
    {%- endif %}
{% endif %}
```